# Health Analysis – Heart Rate and Breathing Rate from IMU Data

Steps:
1. Load the CSV and compute the actual sampling rate from timestamps.
2. Bandpass filter the Z-axis accelerometer for breathing (0.1–0.6 Hz) and heart rate (0.8–3.0 Hz).
3. Find peaks in each filtered signal and compute the rate.
4. Plot the results.

In [ ]:
# Install dependencies (uncomment if running in a fresh Colab environment)
# !pip install numpy pandas scipy matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt, find_peaks, welch
from pathlib import Path

plt.rcParams.update({'figure.dpi': 120, 'axes.grid': True, 'grid.alpha': 0.3})

## 1. Load IMU data

Expected CSV columns: `timestamp_ms, acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z`

Set `CSV_FILES` to point at your recorded trial files.

In [ ]:
CSV_FILES = [
    'imu_trial_1.csv',
    'imu_trial_2.csv',
    'imu_trial_3.csv',
]


def load_trial(path):
    df = pd.read_csv(path)
    df = df.sort_values('timestamp_ms').reset_index(drop=True)
    # Compute sampling rate from the timestamps
    duration_s = (df['timestamp_ms'].iloc[-1] - df['timestamp_ms'].iloc[0]) / 1000.0
    fs = (len(df) - 1) / duration_s
    print(f'Loaded {path}: {len(df):,} samples, duration = {duration_s:.1f} s, fs = {fs:.1f} Hz')
    return df, fs, duration_s


trials = []
for path in CSV_FILES:
    if Path(path).exists():
        trials.append(load_trial(path))
    else:
        print(f'File not found: {path} – skipping')

print(f'\nLoaded {len(trials)} trial(s).')

## 2. Helper functions

In [ ]:
def bandpass(data, low, high, fs, order=4):
    nyq = fs / 2.0
    b, a = butter(order, [low / nyq, high / nyq], btype='band')
    return filtfilt(b, a, data)


def compute_rate(filtered, fs, min_hz, max_hz):
    """Find peaks and return rate in beats/breaths per minute."""
    min_dist = int(fs / max_hz)
    peaks, _ = find_peaks(filtered, distance=min_dist, height=np.std(filtered) * 0.3)
    if len(peaks) < 2:
        return float('nan'), peaks
    intervals_s = np.diff(peaks) / fs
    valid = intervals_s[(intervals_s >= 1 / max_hz) & (intervals_s <= 1 / min_hz)]
    if len(valid) == 0:
        return float('nan'), peaks
    return 60.0 / np.mean(valid), peaks


def dominant_freq(data, fs, fmin, fmax):
    """Return the frequency with the highest PSD power in [fmin, fmax]."""
    freqs, psd = welch(data, fs=fs, nperseg=min(len(data), int(fs * 30)))
    mask = (freqs >= fmin) & (freqs <= fmax)
    if not np.any(mask):
        return float('nan')
    return freqs[mask][np.argmax(psd[mask])]

## 3. Analyse each trial

In [ ]:
BREATHING_LOW, BREATHING_HIGH = 0.1, 0.6
HEART_LOW, HEART_HIGH = 0.8, 3.0

results = []

for idx, (df, fs, duration_s) in enumerate(trials):
    trial_num = idx + 1

    t = df['timestamp_ms'].values / 1000.0  # seconds
    acc_z = df['acc_z'].values
    acc_z = acc_z - acc_z.mean()  # remove DC offset

    # Breathing
    br_signal = bandpass(acc_z, BREATHING_LOW, BREATHING_HIGH, fs)
    br_rate, br_peaks = compute_rate(br_signal, fs, BREATHING_LOW, BREATHING_HIGH)
    br_rate_psd = dominant_freq(acc_z, fs, BREATHING_LOW, BREATHING_HIGH) * 60

    # Heart rate
    hr_signal = bandpass(acc_z, HEART_LOW, HEART_HIGH, fs)
    hr_rate, hr_peaks = compute_rate(hr_signal, fs, HEART_LOW, HEART_HIGH)
    hr_rate_psd = dominant_freq(acc_z, fs, HEART_LOW, HEART_HIGH) * 60

    results.append({
        'trial': trial_num,
        'fs': fs,
        'n_samples': len(df),
        'duration_s': duration_s,
        'breathing_rate': br_rate,
        'breathing_rate_psd': br_rate_psd,
        'heart_rate': hr_rate,
        'heart_rate_psd': hr_rate_psd,
        '_t': t,
        '_acc_z': acc_z,
        '_br_signal': br_signal,
        '_br_peaks': br_peaks,
        '_hr_signal': hr_signal,
        '_hr_peaks': hr_peaks,
    })

    print(f'Trial {trial_num} (fs={fs:.1f} Hz):')
    print(f'  Breathing rate: {br_rate:.1f} br/min (peaks),  {br_rate_psd:.1f} br/min (PSD)')
    print(f'  Heart rate:     {hr_rate:.1f} bpm (peaks),  {hr_rate_psd:.1f} bpm (PSD)')
    print()

## 4. Plot: time-domain traces

In [ ]:
for r in results:
    t = r['_t']
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    fig.suptitle(f'Trial {r["trial"]} – Time-Domain Signals', fontsize=14)

    axes[0].plot(t, r['_br_signal'], color='steelblue', linewidth=0.8,
                 label='Filtered (0.1–0.6 Hz)')
    axes[0].plot(t[r['_br_peaks']], r['_br_signal'][r['_br_peaks']],
                 'rv', markersize=6, label=f'Peaks ({r["breathing_rate"]:.1f} br/min)')
    axes[0].set_ylabel('Acc Z (m/s²)')
    axes[0].set_title('Breathing')
    axes[0].legend(loc='upper right')

    axes[1].plot(t, r['_hr_signal'], color='tomato', linewidth=0.6,
                 label='Filtered (0.8–3.0 Hz)')
    axes[1].plot(t[r['_hr_peaks']], r['_hr_signal'][r['_hr_peaks']],
                 'b^', markersize=5, label=f'Peaks ({r["heart_rate"]:.1f} bpm)')
    axes[1].set_xlabel('Time (s)')
    axes[1].set_ylabel('Acc Z (m/s²)')
    axes[1].set_title('Heart Rate (BCG)')
    axes[1].legend(loc='upper right')

    plt.tight_layout()
    plt.savefig(f'trial_{r["trial"]}_timeseries.png', bbox_inches='tight')
    plt.show()

## 5. Plot: Power Spectral Density

In [ ]:
for r in results:
    fs = r['fs']
    acc_z = r['_acc_z']
    freqs, psd = welch(acc_z, fs=fs, nperseg=min(len(acc_z), int(fs * 30)))

    fig, ax = plt.subplots(figsize=(12, 4))
    ax.semilogy(freqs, psd, color='navy', linewidth=0.9)

    mask_br = (freqs >= BREATHING_LOW) & (freqs <= BREATHING_HIGH)
    ax.fill_between(freqs[mask_br], psd[mask_br], alpha=0.3,
                    color='steelblue', label='Breathing band')

    mask_hr = (freqs >= HEART_LOW) & (freqs <= HEART_HIGH)
    ax.fill_between(freqs[mask_hr], psd[mask_hr], alpha=0.3,
                    color='tomato', label='Heart rate band')

    br_f = r['breathing_rate_psd'] / 60
    hr_f = r['heart_rate_psd'] / 60
    ax.axvline(br_f, color='steelblue', linestyle='--',
               label=f'Breathing: {br_f:.3f} Hz ({r["breathing_rate_psd"]:.1f} br/min)')
    ax.axvline(hr_f, color='tomato', linestyle='--',
               label=f'Heart rate: {hr_f:.3f} Hz ({r["heart_rate_psd"]:.1f} bpm)')

    ax.set_xlim(0, 4)
    ax.set_xlabel('Frequency (Hz)')
    ax.set_ylabel('PSD [(m/s²)²/Hz]')
    ax.set_title(f'Trial {r["trial"]} – Power Spectral Density (Z-axis)')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(f'trial_{r["trial"]}_psd.png', bbox_inches='tight')
    plt.show()

## 6. Summary table

In [ ]:
summary = pd.DataFrame([
    {
        'Trial': r['trial'],
        'fs (Hz)': f"{r['fs']:.1f}",
        'Duration (s)': f"{r['duration_s']:.1f}",
        'Samples': r['n_samples'],
        'Breathing peaks (br/min)': f"{r['breathing_rate']:.1f}",
        'Breathing PSD (br/min)': f"{r['breathing_rate_psd']:.1f}",
        'Heart rate peaks (bpm)': f"{r['heart_rate']:.1f}",
        'Heart rate PSD (bpm)': f"{r['heart_rate_psd']:.1f}",
    }
    for r in results
])

print(summary.to_string(index=False))

if len(results) > 0:
    print('\n--- Mean across all trials ---')
    print(f'Breathing rate: {np.nanmean([r["breathing_rate_psd"] for r in results]):.1f} br/min')
    print(f'Heart rate:     {np.nanmean([r["heart_rate_psd"] for r in results]):.1f} bpm')